In [26]:
# Import or install Sionna
import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

no_preview = True # Toggle to False to use the preview widget

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies

In [27]:
# load and initial scene

scene_path="../my_scene/nycu_carla_v7_add_plane_yf_mirror/nycu_carla_v7_add_plane_yf_mirror.xml"

print("load scene...")
print(f"scene_path: {scene_path}")
scene = sionna.rt.load_scene(scene_path)
#scene = sionna.rt.load_scene(sionna.rt.scene.etoile)
print("load scene success")

print("set material...")
scene.frequency = 2.4e9
new_material = sionna.rt.RadioMaterial(f"my_material_{scene.frequency }",
                            relative_permittivity=5.24,
                            conductivity=0.0462*(scene.frequency/1e9)**0.7822,
                            scattering_pattern='lambertian')
for i, obj in enumerate(scene.objects.values()):
    scene.get(obj.name).radio_material=new_material
print("set material success")

print("current tx and rx objects in the scene:")
print(f"{'='*10} Tx {'='*10} ")
for tx in scene.transmitters:
    print(tx)
print(f"{'='*10} Rx {'='*10} ")
for rx in scene.receivers:
    print(rx)

load scene...
scene_path: ../my_scene/nycu_carla_v7_add_plane_yf_mirror/nycu_carla_v7_add_plane_yf_mirror.xml
load scene success
set material...
set material success
current tx and rx objects in the scene:
========== Tx ========== 
========== Rx ========== 


In [28]:
# set tx and rx parameters

scene.tx_array = PlanarArray(num_rows=2,
                             num_cols=1,
                             pattern="iso",
                             polarization="V")
# scene.rx_array = scene.tx_array
scene.rx_array = PlanarArray(num_rows=4,
                             num_cols=1,
                             pattern="iso",
                             polarization="V")

tx_dict={
    "tx01":{
        "x_pos": -50,
        "y_pos": -50,
        "z_pos": 100
    },
    "tx02":{
        "x_pos": 50,
        "y_pos": 50,
        "z_pos": 100
    }
}

rx_dict={
    "rx01":{
        "x_pos": 0,
        "y_pos": 0,
        "z_pos": 50
    },
    "rx02":{
        "x_pos": 0,
        "y_pos": 50,
        "z_pos": 50
    },
    "rx03":{
        "x_pos": 50,
        "y_pos": 0,
        "z_pos": 50
    }       
}

print("remove old tx and rx objects then add new objects")
for tx in scene.transmitters:
    scene.remove(tx)
for rx in scene.receivers:
    scene.remove(rx)

for tx_name in tx_dict:
    position = [ tx_dict[tx_name]["x_pos"], tx_dict[tx_name]["y_pos"], tx_dict[tx_name]["z_pos"] ]
    scene.add(sionna.rt.Transmitter(name=tx_name, position=position, display_radius=1))
for rx_name in rx_dict:
    position = [ rx_dict[rx_name]["x_pos"], rx_dict[rx_name]["y_pos"], rx_dict[rx_name]["z_pos"] ]
    scene.add(sionna.rt.Receiver(name=rx_name, position=position, display_radius=1))

print("current tx and rx objects in the scene:")
print(f"{'='*10} Tx {'='*10} ")
for tx in scene.transmitters:
    print(tx)
print(f"{'='*10} Rx {'='*10} ")
for rx in scene.receivers:
    print(rx)

remove old tx and rx objects then add new objects
current tx and rx objects in the scene:
========== Tx ========== 
tx01
tx02
========== Rx ========== 
rx01
rx02
rx03


In [29]:
# ray tracing calculation

p_solver=PathSolver()
paths = p_solver(scene=scene,
            max_depth=3,
            los=True,
            specular_reflection=True,
            diffuse_reflection=False,
            refraction=True,
            synthetic_array=False,
            seed=41)

## Property of paths, refer to : https://nvlabs.github.io/sionna/rt/api/paths.html#sionna.rt.Paths

In [30]:
print("tx and rx coordinate")
print(f"{'='*10} Tx {'='*10} ")
print(paths.sources)
print(f"{'='*10} Rx {'='*10} ")
print(paths.targets)

tx and rx coordinate
========== Tx ========== 
[[-50, -50, 100.031],
 [-50, -50, 99.9688],
 [50, 50, 100.031],
 [50, 50, 99.9688]]
========== Rx ========== 
[[0, 0, 50.0937],
 [0, 0, 50.0312],
 [0, 0, 49.9688],
 [0, 0, 49.9063],
 [0, 50, 50.0937],
 [0, 50, 50.0312],
 [0, 50, 49.9688],
 [0, 50, 49.9063],
 [50, 0, 50.0937],
 [50, 0, 50.0312],
 [50, 0, 49.9688],
 [50, 0, 49.9063]]


In [31]:
a ,tau=paths.cir(out_type="numpy", normalize_delays=False)
print(f"num_tx: {paths.num_tx}")
print(f"num_rx: {paths.num_rx}")
print(f"sources.shape: {paths.sources.shape}")
print(f"targets.shape: {paths.targets.shape}")

print(f"a.shape: {a.shape}")
print(f"tau.shape: {tau.shape}")
print(f"doppler.shape: {paths.doppler.shape}")
print(f"interactions.shape: {paths.interactions.shape}")
print(f"phi_r.shape: {paths.phi_r.shape}")
print(f"phi_t.shape: {paths.phi_t.shape}")
print(f"theta_r.shape: {paths.theta_r.shape}")
print(f"theta_t.shape: {paths.theta_t.shape}")


num_tx: 2
num_rx: 3
sources.shape: (3, 4)
targets.shape: (3, 12)
a.shape: (3, 4, 2, 2, 7, 1)
tau.shape: (3, 4, 2, 2, 7)
doppler.shape: (3, 4, 2, 2, 7)
interactions.shape: (3, 3, 4, 2, 2, 7)
phi_r.shape: (3, 4, 2, 2, 7)
phi_t.shape: (3, 4, 2, 2, 7)
theta_r.shape: (3, 4, 2, 2, 7)
theta_t.shape: (3, 4, 2, 2, 7)


In [32]:
print(f"type num_tx: {type(paths.num_tx)}")
print(f"type num_rx: {type(paths.num_rx)}")
print(f"type sources: {type(paths.sources)}")
print(f"type targets: {type(paths.targets)}")

print(f"type a: {type(a)}")
print(f"type tau: {type(tau)}")
print(f"type doppler: {type(paths.doppler)}")
print(f"type interactions: {type(paths.interactions)}")
print(f"type phi_r: {type(paths.phi_r)}")
print(f"type phi_t: {type(paths.phi_t)}")
print(f"type theta_r: {type(paths.theta_r)}")
print(f"type theta_t: {type(paths.theta_t)}")


type num_tx: <class 'int'>
type num_rx: <class 'int'>
type sources: <class 'mitsuba.Point3f'>
type targets: <class 'mitsuba.Point3f'>
type a: <class 'numpy.ndarray'>
type tau: <class 'numpy.ndarray'>
type doppler: <class 'drjit.cuda.ad.TensorXf'>
type interactions: <class 'drjit.cuda.ad.TensorXu'>
type phi_r: <class 'drjit.cuda.ad.TensorXf'>
type phi_t: <class 'drjit.cuda.ad.TensorXf'>
type theta_r: <class 'drjit.cuda.ad.TensorXf'>
type theta_t: <class 'drjit.cuda.ad.TensorXf'>


In [33]:
a, tau = paths.cir(out_type="numpy", normalize_delays=False)
num_rx_antennas=paths.targets.shape[1]//paths.num_rx
num_tx_antennas=paths.sources.shape[1]//paths.num_tx
print(f"num rx:{paths.num_rx}, num_rx_antennas:{num_rx_antennas}")
print(f"num tx:{paths.num_tx}, num_rx_antennas:{num_tx_antennas}")

# set rx and tx name list
rx_name_list=[]
tx_name_list=[]
for rx_name in scene.receivers:
    rx_name_list.append(rx_name)
for tx_name in scene.transmitters:
    tx_name_list.append(tx_name)



for tx_idx in range(paths.num_tx):
    for rx_idx in range(paths.num_rx):
        print(f"{'='*60}")
        print(f"TX {tx_name_list[tx_idx]} -> RX {rx_name_list[rx_idx]}")
        print(f"{'='*60}")
        for tx_ant_idx in range(num_tx_antennas):
            for rx_ant_idx in range(num_rx_antennas):
                tx_ant_pos=(paths.sources[0][tx_idx*num_tx_antennas+tx_ant_idx], 
                            paths.sources[1][tx_idx*num_tx_antennas+tx_ant_idx], 
                            paths.sources[2][tx_idx*num_tx_antennas+tx_ant_idx])
                            
                rx_ant_pos=(paths.targets[0][rx_idx*num_rx_antennas+rx_ant_idx], 
                            paths.targets[1][rx_idx*num_rx_antennas+rx_ant_idx], 
                            paths.targets[2][rx_idx*num_rx_antennas+rx_ant_idx])
                print(f"-"*40)
                print(f"tx_ant_idx: {tx_ant_idx} -> rx_ant_idx {rx_ant_idx}")
                print(f"tx_ant_pos: {tx_ant_pos}")
                print(f"rx_ant_pos: {rx_ant_pos}")
                print(f"-"*40)
                print(f"a: {a[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :, 0]}")
                print(f"tau: {tau[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]}")
                print(f"doppler: {paths.doppler[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]}")
                print(f"phi_r: {paths.phi_r[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]}")
                print(f"phi_t: {paths.phi_t[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]}")
                print(f"theta_r: {paths.theta_r[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]}")
                print(f"theta_t: {paths.theta_t[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]}")




# for rx
# rx_idx = 0
# for rx_name in scene.receivers:
#     print(f"\n{'='*60}")
#     print(f"RX {rx_idx}: {rx_name}")
#     print(f"{'='*60}")
#     tx_idx = 0
#     for tx_name in scene.transmitters:
#         print(f"\n  TX {tx_idx}: {tx_name} -> RX {rx_idx}: {rx_name}")
#         print(f"  {'-'*56}")
#         for rx_ant_idx in range(num_rx_antennas // paths.num_rx):
#             for tx_ant_idx in range(num_tx_antennas // paths.num_tx)
#             # 提取該 TX-RX 配對的資料
#             a_tx_rx = a[rx_idx, rx_ant_idx, tx_idx, tx_antenna_idx, :, :]
#             tau_tx_rx = tau[:, :, tx_idx, :, :]
#             doppler_tx_rx = paths.doppler[:, :, tx_idx, :, :]
#             phi_r_tx_rx = paths.phi_r[:, :, tx_idx, :, :]
#             phi_t_tx_rx = paths.phi_t[:, :, tx_idx, :, :]
#             theta_r_tx_rx = paths.theta_r[:, :, tx_idx, :, :]
#             theta_t_tx_rx = paths.theta_t[:, :, tx_idx, :, :]
        
#         print(f"  a shape: {a_tx_rx.shape}")
#         print(f"  tau shape: {tau_tx_rx.shape}")
#         print(f"  doppler shape: {doppler_tx_rx.shape}")
        
#         tx_idx += 1
    
#     rx_idx += 1

num rx:3, num_rx_antennas:4
num tx:2, num_rx_antennas:2
TX tx01 -> RX rx01
----------------------------------------
tx_ant_idx: 0 -> rx_ant_idx 0
tx_ant_pos: (-50.0, -50.0, 100.0312271118164)
rx_ant_pos: (0.0, 0.0, 50.093685150146484)
----------------------------------------
a: [-2.4840640e-05+4.5506855e-08j  1.1453269e-04-8.2372298e-06j
  0.0000000e+00+0.0000000e+00j  0.0000000e+00+0.0000000e+00j
  0.0000000e+00+0.0000000e+00j  0.0000000e+00+0.0000000e+00j
  0.0000000e+00+0.0000000e+00j]
tau: [ 5.5353036e-07  2.8875476e-07 -1.0000000e+00 -1.0000000e+00
 -1.0000000e+00 -1.0000000e+00 -1.0000000e+00]
doppler: [0, 0, 0, 0, 0, 0, 0]
phi_r: [-2.35619, -2.35619, 0, 0, 0, 0, 0]
phi_t: [0.785398, 0.785399, 0, 0, 0, 0, 0]
theta_r: [2.7014, 0.955906, 0, 0, 0, 0, 0]
theta_t: [2.7014, 2.18569, 0, 0, 0, 0, 0]
----------------------------------------
tx_ant_idx: 0 -> rx_ant_idx 1
tx_ant_pos: (-50.0, -50.0, 100.0312271118164)
rx_ant_pos: (0.0, 0.0, 50.031227111816406)
-------------------------------

In [37]:

# print(f"{'='*10} Tx {'='*10} ")
# for tx in scene.transmitters:
#     antenna_id=tx
#     position=scene.get(antenna_id).position
#     orientation=scene.get(antenna_id).orientation
#     look_at=scene.get(antenna_id).look_at
#     velocity=scene.get(antenna_id).velocity
#     data={
#         "id":tx,
#         "type":"antenna",
#         "position":position,
#         "orientation":orientation,
#         "look_at":look_at,
#         "velocity":velocity
#     }
#     for key in data:
#         print(f"type: {type(data[key])}")
#         print(f"data: {data[key]}")
print(f"{'='*10} Rx {'='*10} ")
for rx in scene.receivers:
    antenna_id=rx
    position=scene.get(antenna_id).position
    orientation=scene.get(antenna_id).orientation
    # look_at=scene.get(antenna_id).look_at
    velocity=scene.get(antenna_id).velocity
    position=np.array(position).flatten().astype(float).tolist()
    orientation=np.array(orientation).flatten().astype(float).tolist()
    # look_at=np.array(look_at).flatten().astype(float).tolist()
    velocity=np.array(velocity).flatten().astype(float).tolist()
    data={
        "id":rx,
        "type":"antenna",
        "position":position,
        "orientation":orientation,
        "velocity":velocity
    }
    for key in data:
        print(f"type: {type(data[key])}")
        print(f"data: {data[key]}")

========== Rx ========== 
type: <class 'str'>
data: rx01
type: <class 'str'>
data: antenna
type: <class 'list'>
data: [0.0, 0.0, 50.0]
type: <class 'list'>
data: [0.0, 0.0, 0.0]
type: <class 'list'>
data: [0.0, 0.0, 0.0]
type: <class 'str'>
data: rx02
type: <class 'str'>
data: antenna
type: <class 'list'>
data: [0.0, 50.0, 50.0]
type: <class 'list'>
data: [0.0, 0.0, 0.0]
type: <class 'list'>
data: [0.0, 0.0, 0.0]
type: <class 'str'>
data: rx03
type: <class 'str'>
data: antenna
type: <class 'list'>
data: [50.0, 0.0, 50.0]
type: <class 'list'>
data: [0.0, 0.0, 0.0]
type: <class 'list'>
data: [0.0, 0.0, 0.0]


In [212]:
import pickle
import json
import numpy as np
from pathlib import Path

def save_paths_data(base_dir, timestamp, scene, paths, extra_metadata=None):
    """
    儲存單一時刻的 Sionna paths 數據
    
    Parameters:
    -----------
    base_dir : str
        基礎目錄路徑
    timestamp : float
        時間戳記
    paths : sionna paths object
        包含 sources, targets, doppler, phi_r, phi_t, theta_r, theta_t
    a : np.ndarray
        複數振幅陣列
    tau : np.ndarray
        時延陣列
    tx_name_list : list
        TX 名稱列表，例如 ['tx01', 'tx02']
    rx_name_list : list
        RX 名稱列表，例如 ['rx01', 'rx02', 'rx03']
    num_tx_antennas : int
        每個 TX 的天線數量
    num_rx_antennas : int
        每個 RX 的天線數量
    extra_metadata : dict (optional)
        額外的元數據（例如 frequency, scenario 等）
    """
    
    base_path = Path(base_dir)
    base_path.mkdir(parents=True, exist_ok=True)
    a, tau = paths.cir(out_type="numpy", normalize_delays=False)
    num_rx_antennas=paths.targets.shape[1]//paths.num_rx
    num_tx_antennas=paths.sources.shape[1]//paths.num_tx
    print(f"num rx:{paths.num_rx}, num_rx_antennas:{num_rx_antennas}")
    print(f"num tx:{paths.num_tx}, num_rx_antennas:{num_tx_antennas}")
    
    # set rx and tx name list
    rx_name_list=[]
    tx_name_list=[]
    for rx_name in scene.receivers:
        rx_name_list.append(rx_name)
    for tx_name in scene.transmitters:
        tx_name_list.append(tx_name)


    
    # 轉換 sources 和 targets 為 numpy array
    sources_np = paths.sources.numpy()  # shape: (3, num_tx * num_tx_antennas)
    targets_np = paths.targets.numpy()  # shape: (3, num_rx * num_rx_antennas)
    
    # 轉換 drjit TensorXf 為 numpy array
    doppler_np = paths.doppler.numpy()
    phi_r_np = paths.phi_r.numpy()
    phi_t_np = paths.phi_t.numpy()
    theta_r_np = paths.theta_r.numpy()
    theta_t_np = paths.theta_t.numpy()
    
    # 創建該時刻的 metadata
    meta = {
        'timestamp': timestamp,
        'frequency': scene.frequency,
        'num_tx': paths.num_tx,
        'num_rx': paths.num_rx,
        'num_tx_antennas': num_tx_antennas,
        'num_rx_antennas': num_rx_antennas,
        'tx_names': tx_name_list,
        'rx_names': rx_name_list,
        'tx_ant_positions': {},
        'rx_ant_positions': {}
    }
    
    # 儲存天線位置
    for tx_idx in range(paths.num_tx):
        tx_positions = []
        tx_ant_dict={}
        for tx_ant_idx in range(num_tx_antennas):   
            ant_global_idx = tx_idx * num_tx_antennas + tx_ant_idx
            pos = (
                float(sources_np[0, ant_global_idx]),
                float(sources_np[1, ant_global_idx]),
                float(sources_np[2, ant_global_idx])
            )
            tx_ant_dict[tx_ant_idx]=pos
        meta['tx_ant_positions'][tx_name_list[tx_idx]]=tx_ant_dict
    
    for rx_idx in range(paths.num_rx):
        rx_positions = []
        rx_ant_dict={}
        for rx_ant_idx in range(num_rx_antennas):
            ant_global_idx = rx_idx * num_rx_antennas + rx_ant_idx
            pos = (
                float(targets_np[0, ant_global_idx]),
                float(targets_np[1, ant_global_idx]),
                float(targets_np[2, ant_global_idx])
            )
            rx_ant_dict[rx_ant_idx]=pos
        meta['rx_ant_positions'][rx_name_list[rx_idx]]=rx_ant_dict
    
    # 加入其他 metadata
    if extra_metadata:
        for key, value in extra_metadata.items():
            if key not in meta:  # 避免覆蓋已有的 key
                meta[key] = value
    
    # 儲存 metadata 為 JSON（每個時刻一個檔案）
    metadata_file = base_path / f"metadata_frame{timestamp}.json"
    with open(metadata_file, 'w') as f:
        json.dump(meta, f, indent=2)
    
    # 按照 TX-RX 對儲存 pickle 檔案
    for tx_idx in range(paths.num_tx):
        for rx_idx in range(paths.num_rx):
            tx_name = tx_name_list[tx_idx]
            rx_name = rx_name_list[rx_idx]
            
            # 創建目錄 {tx_name}-{rx_name}
            link_dir = base_path / f"{tx_name}-{rx_name}"
            link_dir.mkdir(exist_ok=True)
            
            for tx_ant_idx in range(num_tx_antennas):
                for rx_ant_idx in range(num_rx_antennas):
                    
                    # 提取該天線對的數據（全部轉為 numpy array）
                    data = {
                        'timestamp': timestamp,
                        'tx_name': tx_name,
                        'rx_name': rx_name,
                        'tx_ant_idx': tx_ant_idx,
                        'rx_ant_idx': rx_ant_idx,
                        # CIR與其他通訊參數（確保都是 numpy array）
                        'a': np.array(a[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :, 0]),
                        'tau': np.array(tau[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]),
                        'doppler': np.array(doppler_np[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]),
                        'phi_r': np.array(phi_r_np[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]),
                        'phi_t': np.array(phi_t_np[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]),
                        'theta_r': np.array(theta_r_np[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]),
                        'theta_t': np.array(theta_t_np[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :])
                    }
                    
                    # 檔名格式: tx{ant_idx}-rx{ant_idx}_t{timestamp}.pkl
                    filename = f"tx_ant{tx_ant_idx}-rx_ant{rx_ant_idx}_frame{timestamp}.pkl"
                    filepath = link_dir / filename
                    
                    with open(filepath, 'wb') as f:
                        pickle.dump(data, f)
    

In [213]:
save_paths_data("dataset_test", timestamp=1, paths=paths, frequency=2.4e9,extra_metadata=None)

num rx:3, num_rx_antennas:4
num tx:2, num_rx_antennas:2
✓ 已儲存時刻 t=1s 的數據


In [234]:
with open("../dataset/scenario_003/channel/frame_00010/tx01-rx_94/tx_ant0-rx_ant0_frame00010.pkl", 'rb') as f:
    data = pickle.load(f)
for key in data:
    print(f"{key}: {data[key]}")

timestamp: 10
tx_name: tx01
rx_name: rx_94
tx_ant_idx: 0
rx_ant_idx: 0
a: [-4.1568729e-08+8.9086907e-09j  8.9113840e-08-3.3281534e-08j
  1.0530181e-05-2.4324934e-06j  8.4192656e-08-6.9779489e-08j
 -1.0226570e-07-1.9092116e-07j  8.7705303e-08-3.4908364e-08j
  5.2503641e-09+9.5123829e-09j -2.1587016e-07+4.8726912e-07j
 -9.5566488e-09+1.0653965e-07j  1.6528332e-08-1.9761133e-09j
 -1.3386529e-06-2.0211610e-07j  4.2417912e-07+3.2867565e-06j
  2.5798806e-06+1.6578739e-06j  1.7930781e-06-2.1629616e-07j
  4.1868082e-09+1.5075695e-08j  5.8549445e-09+9.6377384e-10j
 -2.4922515e-09+6.0943828e-10j  7.8711455e-09-3.9012971e-09j
 -5.7871010e-09-3.1170799e-08j -9.8044239e-09+3.3724665e-09j
  5.9331260e-08+3.8092434e-08j -1.0890899e-08-1.2565951e-08j
  1.6332213e-11+6.7618167e-10j  1.2031752e-08-2.6352296e-09j
  1.1089514e-08-1.3465208e-08j  5.4236457e-06+1.7323478e-06j
  1.8513323e-05+1.9010660e-05j  0.0000000e+00+0.0000000e+00j
  0.0000000e+00+0.0000000e+00j  0.0000000e+00+0.0000000e+00j
  0.0000000

In [227]:
a[0][0][0][0][0]

array([0.00011453-8.23723e-06j], dtype=complex64)

In [228]:
tau[0][0][0][0][0]

2.8875476e-07

In [229]:
with open("dataset_test/tx02-rx03/tx_ant1-rx_ant3_frame1.pkl", 'rb') as f:
    data = pickle.load(f)
for key in data:
    print(data[key])

1
tx02
rx03
1
3
[-4.7029712e-06-4.9802964e-07j -1.2737226e-04-5.9275248e-05j
  1.8252485e-05+1.9743578e-05j  1.5994168e-05-1.3502185e-05j
  0.0000000e+00+0.0000000e+00j  0.0000000e+00+0.0000000e+00j
  0.0000000e+00+0.0000000e+00j]
[ 6.6831348e-07  2.3601278e-07  5.2701591e-07  5.2982642e-07
 -1.0000000e+00 -1.0000000e+00 -1.0000000e+00]
[0. 0. 0. 0. 0. 0. 0.]
[-1.5336937  1.5707964  1.5707964  1.9029976  0.         0.
  0.       ]
[-1.5539707 -1.5707964 -1.5707954 -1.8710556  0.         0.
  0.       ]
[2.4159126  0.78477395 2.8195908  2.8040514  0.         0.
 0.        ]
[2.4159102 2.3568187 2.8195922 2.8040512 0.        0.        0.       ]


In [230]:
a[2][3][1][1][0]

array([-4.702971e-06-4.9802964e-07j], dtype=complex64)

In [231]:
tau[2][3][1][1][0]

6.683135e-07

In [232]:
data["tau"][0]

6.683135e-07